# Analiza Danych

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from google.cloud import bigquery
from google.oauth2 import service_account
credentials = service_account.Credentials.from_service_account_file("key.json")
client = bigquery.Client(credentials=credentials, project=credentials.project_id)

## Część 1:
Analiza zmian zjawisk w czasie porządkuje informacje i pozwala na wyciągnięcie bardziej szczegółowych wniosków, np. trendów ze składowymi okresowymi.Trend to tendencja rozwojowa, która wskazuje ogólny kierunek rozwoju zjawiska. Rozwój zjawiska rozumiany jest jako systematyczne zmiany, jakim podlega to zjawisko. Rozróżniany jest m.in. trend liniowy i nieliniowy.Składowe okresowe to regularne odchylenia od trendu. Wahania cykliczne charakteryzują się długookresowymi, rytmicznymi odchyleniami. Wahania sezonowe są krótkookresowe i odzwierciedlają wpływ zachowań wynikający z kalendarza.
#### Wyznacz średnią kroczącą i odchylenie standardowe kroczące (dla sensownie wybranego okna czasowego), aby zobaczyć, jak te zjawiska zmieniają się w czasie. Obliczenia wykonaj dla:
1. temperatury,
2. opadów,
3. prędkości wiatru,
4. 3 innych różnych informacji, które uznasz za istotne (uwzględnij co najmniej jedną zmienną związaną z rolnictwem).

Dobierz sposób reprezentacji danych w czasie w taki sposób, aby możliwe było ich porównanie między sobą oraz interpretacja zmian. Przygotuj wykresy umożliwiające analizę zmian w czasie. Pamiętaj o ich czytelności (typ wykresu, tytuł wykresu, podpisy osi, odpowiednie zakresy osi, itp.). Przeanalizuj otrzymane wizualizacje.

In [2]:
query = """
        SELECT stn, wban, temp, dewp, slp, stp, visib, wdsp, prcp, sndp, `date`,mxpsd, gust,max, min
        FROM `bigquery-public-data.noaa_gsod.gsod2020` AS gsod

            UNION ALL
              SELECT stn, wban, temp, dewp,slp, stp, visib, wdsp, prcp, sndp, `date`,mxpsd, gust,max, min
        FROM `bigquery-public-data.noaa_gsod.gsod2022` AS gsod

            UNION ALL
              SELECT stn, wban, temp, dewp, slp, stp, visib, wdsp, prcp, sndp, `date`,mxpsd, gust, max, min
        FROM `bigquery-public-data.noaa_gsod.gsod2024` AS gsod

        """

df = client.query(query).to_dataframe()


In [3]:
df['wban'] = df['wban'].replace('99999', None)
df['temp'] = df['temp'].replace(9999.9, None).astype(np.float64)
df['slp'] = df['slp'].replace(9999.9, None).astype(np.float64)
df['stp'] = df['stp'].replace(9999.9, None).astype(np.float64)
df['visib'] = df['visib'].replace(999.9, None).astype(np.float64)
df['wdsp'] = df['wdsp'].replace('999.9', None).astype(np.float64)
df['dewp'] = df['dewp'].replace(9999.9, None).astype(np.float64)
df['prcp'] = df['prcp'].replace(99.99, None).astype(np.float64)
df['sndp'] = df['sndp'].replace(999.9, None).astype(np.float64)
df['mxpsd'] = df['mxpsd'].replace('999.9', None).astype(np.float64)
df['gust'] = df['gust'].replace(999.9, None).astype(np.float64)
df['max'] = df['max'].replace(9999.9, None).astype(np.float64)
df['min'] = df['min'].replace(9999.9, None).astype(np.float64)
df = df.drop_duplicates()
df['date'] = pd.to_datetime(df['date'])
df['month'] = pd.to_datetime(df['date']).dt.month
df.to_csv('data/data_3_years.csv', index=False)

In [4]:
stations = pd.read_csv('data/zad4_1.csv')
pd.read_csv('data/data_3_years.csv').merge(stations, how='inner', left_on=['stn', 'wban'], right_on=['usaf', 'wban']).drop(columns=['usaf'])\
    .to_csv('data/3_years_data.csv', index=False)

/tmp/ipykernel_71982/2134626873.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('data/data_3_years.csv').merge(stations, how='inner', left_on=['stn', 'wban'], right_on=['usaf', 'wban']).drop(columns=['usaf'])\


In [5]:
df = pd.read_csv('data/3_years_data.csv')

In [6]:
imputated = df[['stn', 'dewp','wban','wdsp', 'prcp', 'date', 'name', 'country', 'month']].copy()

imputated['mean_prcp'] = imputated.groupby(['name', 'month'])['prcp'].transform(lambda g: g.fillna(g.mean()))
imputated['mean_wdsp'] = imputated.groupby(['name', 'month'])['wdsp'].transform(lambda g: g.fillna(g.mean()))
imputated['mean_dewp'] = imputated.groupby(['name', 'month'])['dewp'].transform(lambda g: g.fillna(g.mean()))

imputated['mean_prcp'] = imputated.groupby(['country', 'month'])['prcp'].transform(lambda g: g.fillna(g.mean()))
imputated['mean_wdsp'] = imputated.groupby(['country', 'month'])['wdsp'].transform(lambda g: g.fillna(g.mean()))
imputated['mean_dewp'] = imputated.groupby(['country', 'month'])['dewp'].transform(lambda g: g.fillna(g.mean()))

imputated['mean_prcp'] = imputated.groupby([ 'month'])['prcp'].transform(lambda g: g.fillna(g.mean()))
imputated['mean_wdsp'] = imputated.groupby(['month'])['wdsp'].transform(lambda g: g.fillna(g.mean()))
imputated['mean_dewp'] = imputated.groupby(['month'])['dewp'].transform(lambda g: g.fillna(g.mean()))


In [7]:
pd.DataFrame({'Liczba braków': imputated.isnull().sum(), 'Procent (%)': imputated.isnull().mean() * 100}).loc[['prcp', 'mean_prcp', 'wdsp', 'mean_wdsp','dewp', 'mean_dewp']]

df['prcp'] = imputated['mean_prcp']
df['wdsp'] = imputated['mean_wdsp']
df['dewp'] = imputated['mean_dewp']

df.to_csv('data/data_3_years_imputed.csv', index=False)

### Właściwy kod zadania


In [2]:
df = pd.read_csv('data/data_3_years_imputed.csv')

variables = ['temp', 'prcp', 'wdsp', 'dewp', 'visib', 'mxpsd']

for var in variables:
    df[f'{var}_mean'] = df[var].rolling(window=30, min_periods=15).mean()
    df[f'{var}_std'] = df[var].rolling(window=30, min_periods=15).std()

In [9]:
df.head()

,stn,wban,temp,dewp,slp,stp,visib,wdsp,prcp,sndp,...,prcp_mean,prcp_std,wdsp_mean,wdsp_std,dewp_mean,dewp_std,visib_mean,visib_std,mxpsd_mean,mxpsd_std
0,869660,NaN,54.1,52.7,1015.1,903.7,NaN,3.5,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,877910,NaN,72.5,56.4,1008.9,8.1,18.6,12.1,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,722688,93034.0,48.1,34.9,NaN,888.0,8.9,12.4,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,726777,94055.0,47.1,31.9,1000.8,897.6,10.0,17.8,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,817450,NaN,83.5,75.4,1011.6,1.4,NaN,3.3,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
sns.set_theme(style="whitegrid", context="talk")

# 2. Przygotowanie siatki wykresów (3 wiersze, 2 kolumny)
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(16, 18), sharex=True)
fig.suptitle('Analiza zmienności pogody w czasie (okno 30-dniowe)', fontsize=20, y=0.98)

# Spłaszczenie tablicy osi dla łatwiejszej iteracji
axes = axes.flatten()

# Zmienne i ich polskie odpowiedniki do tytułów
variables = ['temp', 'prcp', 'wdsp', 'dewp', 'visib', 'mxpsd']
titles = [
    'Temperatura [°C/F]',
    'Opady [mm/cale]',
    'Prędkość wiatru',
    'Punkt rosy',
    'Widzialność',
    'Maksymalna prędkość wiatru'
]

# 3. Rysowanie wykresów w pętli
for i, var in enumerate(variables):
    ax = axes[i]

    # Surowe dane dzienne (jasnoszare tło)
    sns.lineplot(data=df, x=df.index, y=var, ax=ax, color='gray', alpha=0.3, linewidth=1, label='Dane dzienne')

    # Średnia krocząca (wyraźna linia)
    sns.lineplot(data=df, x=df.index, y=f'{var}_mean', ax=ax, color='darkblue', linewidth=2.5, label='Średnia krocząca')

    # Wstęga odchylenia standardowego wokół średniej
    ax.fill_between(
        df.index,
        df[f'{var}_mean'] - df[f'{var}_std'],
        df[f'{var}_mean'] + df[f'{var}_std'],
        color='cornflowerblue', alpha=0.3, label='+/- 1 Odchylenie Std'
    )

    # Formatowanie pojedynczego wykresu
    ax.set_title(titles[i], fontweight='bold')
    ax.set_xlabel('Data')
    ax.set_ylabel('Wartość')
    ax.legend(loc='upper left', fontsize=10)

# 4. Poprawa czytelności układu
plt.tight_layout()
fig.subplots_adjust(top=0.94) # Zrobienie miejsca na główny tytuł
plt.show()

## Misja dodatkowa -Szeregi czasowe:

Analiza szeregów czasowych ma na celu zbadanie zjawiska na podstawie zachodzących zmian w czasie u mierzalnych wartości. Model buduje się w oparciu o część systematyczną, tj. trendy oraz wahania cykliczne i sezonowe. Ewentualne szumy i wahania przypadkowe nie są uwzględniane.

Wykorzystaj co najmniej dwie różne metody analizy szeregów czasowych, aby zbadać 5 przypadków rozważanych w części 1.
 Możesz wykorzystać m.in.: średnią ruchomą lub inne metody wygładzania, analizę trendu (np. regresję względem czasu), analizę zmian między kolejnymi okresami (np. różnice lub zmiany procentowe), dekompozycję szeregu czasowego lub inne wybrane podejście. Dobierz metody adekwatnie do charakteru danych i liczby dostępnych obserwacji.

Przygotuj wykresy umożliwiające porównanie wyników uzyskanych różnymi metodami. Zwróć uwagę na czytelność wizualizacji oraz możliwość interpretacji wyników. Przedstaw wnioski.


## Część 3:

Analiza regresji pozwala na przewidzenie nieznanych wartości na podstawie innych, które są znane. Jest to metoda statystyczna, która opisuje współzmienności kilku zmiennych przez dopasowanie do nich odpowiedniej funkcji. Model regresji liniowej zakłada liniową zależność między zmienną niezależną (np. czasem), a zmienną zależną (np. temperaturą).

Przygotuj dane treningowe w interesującym Cię okresie czasu oraz dane testowe. Jednostką analizy powinny być obserwacje porównywalne w czasie. Zastosuj analizę regresji, aby przewidzieć wartości w kolejnych okresach następujących po wybranym okresie treningowym. Użyj modelu regresji liniowej, gdzie zmienną niezależną będzie czas, a zmienną zależną:

1. temperatura,

2. opady,

3. prędkość wiatru,

4.  3 inne różne informacje, które uznasz za istotne, w tym co najmniej jedna zmienna związana z rolnictwem.

Przygotuj wykresy w celu porównania otrzymanych wyników. Pamiętaj o ich czytelności. Przeanalizuj otrzymane wizualizacje. Zastanów się nad uzyskanymi wynikami np. czy model liniowy dobrze oddaje kierunek zmian, czy wyniki są spójne dla różnych zmiennych, czy zmiana zakresu danych treningowych wpływa na wynik modelu, czy uzyskane prognozy są realistyczne w kontekście analizowanego zjawiska.

## Misja dodatkowa - Regresja wielomianowa:

Brak polecenia

## Część 5:

Porównanie zmian badanych zjawisk w różnych krajach pozwala lepiej zrozumieć wpływ czynników geograficznych i środowiskowych.

Wybierz minimum 2 kraje, które ze sobą porównasz. Kryterium wyboru powinno umożliwiać zauważenie różnic (np. położenie geograficzne, warunki klimatyczne, struktura produkcji rolnej). Preferowany jest taki wybór, aby móc porównać kraje położone w różnych częściach świata (np. jeden z półkuli północnej, drugi z południowej). Przygotuj porównywalne dane dla wybranych krajów w tym samym okresie czasu.

Zbuduj model regresji liniowej dla każdego kraju, w którym zmienną niezależną będzie czas, a zmiennymi zależnymi:

5.1. temperatura,

5.2. opady,

5.3. prędkość wiatru,

5.4. 3 inne różne informacje, które uznasz za istotne, w tym co najmniej jedna zmienna związana z rolnictwem..

Przygotuj wykresy w celu porównania otrzymanych wyników. Pamiętaj o ich czytelności (typ wykresu, tytuł wykresu, podpisy osi, odpowiednie zakresy osi, itp.). Przeanalizuj otrzymane wizualizacje. Zastanów się czy obserwowane zależności są podobne dla różnych krajów, jakie czynniki mogą tłumaczyć różnice, czy wyniki są spójne z charakterystyką wybranych krajów, czy porównanie jest wiarygodne przy przyjętym sposobie reprezentacji danych.